# 03 · ResNet50 training

Transfer-learning classifier built from scratch on the leaf-grouped splits from
notebook 02.

```
Input 224×224×3 (resnet50.preprocess_input applied by the pipeline)
  → ResNet50 (ImageNet weights, include_top=False)
  → GlobalAveragePooling2D → Dropout → Dense (ReLU) → Dense softmax (num_classes)
```

* **Stage 1:** the backbone is frozen and only the head is trained (Adam, lr 1e-3).
* **Stage 2:** `conv5_x`, the last residual stage, is unfrozen with BatchNorm kept
  frozen, and training continues (Adam, lr 1e-5).
* **Callbacks:** ModelCheckpoint saves the best `val_accuracy` to
  `models/resnet50/best_resnet50.keras`. EarlyStopping and ReduceLROnPlateau watch
  `val_loss`.

This notebook verifies the model and runs a **smoke test**, which is a few steps
only and not a performance measurement. Set `RUN_FULL_TRAINING = True` to train
for real, or run `python scripts/train_resnet50.py` in the background.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

# Chart styling: light surface, recessive axes, fixed categorical order.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]   # train, val/valid, test
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#b5b4ad", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
})

import time
import tensorflow as tf
from tensorflow import keras

from src.preprocessing import config, pipeline, splits
from src.model import resnet50

tf.keras.utils.set_random_seed(config.SEED)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU") or "none (CPU)")
print(json.dumps(resnet50.TRAIN, indent=2))

TensorFlow 2.21.0 | GPUs: none (CPU)
{
  "dropout": 0.3,
  "dense_units": 256,
  "stage1_epochs": 10,
  "stage1_lr": 0.001,
  "stage2_epochs": 15,
  "stage2_lr": 1e-05,
  "finetune_from": "conv5_block1_1_conv",
  "early_stopping_patience": 5,
  "reduce_lr_patience": 2,
  "reduce_lr_factor": 0.2,
  "min_lr": 1e-07
}


## 1 · Classes come from the dataset

In [2]:
assert (config.SPLITS_DIR / "train.csv").exists(), "run notebook 02 first"
class_names = resnet50.load_class_names()
num_classes = len(class_names)
train_labels = set(splits.load_split("train")["label"])
assert train_labels == set(class_names), "class list does not match training data"

resnet50.save_class_names(class_names)
reloaded = np.load(resnet50.CLASS_NAMES_PATH)
assert list(reloaded) == class_names
print(f"num_classes = {num_classes} (from {config.SPLITS_DIR / 'class_names.json'})")
print(f"saved {resnet50.CLASS_NAMES_PATH}")

num_classes = 38 (from C:\Users\H P\Desktop\Study\7 - Sem\Project\new smart crop care\data\splits\class_names.json)
saved C:\Users\H P\Desktop\Study\7 - Sem\Project\new smart crop care\models\resnet50\class_names.npy


## 2 · Build the model (stage 1: frozen backbone)

In [3]:
model, backbone = resnet50.build_model(num_classes)
resnet50.compile_model(model, resnet50.TRAIN["stage1_lr"])
model.summary()
print("Stage 1:", resnet50.count_params(model))
assert not backbone.trainable
assert model.output_shape == (None, num_classes)

Model: "resnet50_plant_disease"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,122,022 (92.02 MB)

 Trainable params: 534,310 (2.04 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Stage 1: {'trainable': 534310, 'non_trainable': 23587712}


## 3 · Stage 2 layer selection

In [4]:
probe, probe_backbone = resnet50.build_model(num_classes)
start = resnet50.unfreeze_upper_layers(probe_backbone)
trainable = [l for l in probe_backbone.layers if l.trainable]
bn_trainable = [l.name for l in trainable if isinstance(l, keras.layers.BatchNormalization)]
print(f"Backbone layers: {len(probe_backbone.layers)}; fine-tuned from index {start} "
      f"({probe_backbone.layers[start].name}); trainable layers: {len(trainable)}")
print("first / last trainable:", trainable[0].name, "/", trainable[-1].name)
print("BatchNorm layers left trainable:", bn_trainable)
resnet50.compile_model(probe, resnet50.TRAIN["stage2_lr"])
print("Stage 2:", resnet50.count_params(probe))
assert not bn_trainable and all(not l.trainable for l in probe_backbone.layers[:start])
del probe, probe_backbone

Backbone layers: 175; fine-tuned from index 143 (conv5_block1_1_conv); trainable layers: 22
first / last trainable: conv5_block1_1_conv / conv5_block3_out
BatchNorm layers left trainable: []
Stage 2: {'trainable': 15487782, 'non_trainable': 8634240}


## 4 · Forward pass on real data (untrained head)

In [5]:
x, y = next(iter(pipeline.make_dataset("val", batch_size=8)))
probs = model.predict(x, verbose=0)
print("output shape:", probs.shape, "| row sums:", np.round(probs.sum(axis=1), 4))
assert probs.shape == (8, num_classes) and np.allclose(probs.sum(axis=1), 1, atol=1e-4)

output shape: (8, 38) | row sums: [1. 1. 1. 1. 1. 1. 1. 1.]


## 5 · Estimated training time on this machine

In [6]:
def time_steps(m, steps=5, batch_size=config.BATCH_SIZE):
    ds = pipeline.make_dataset("train", batch_size=batch_size).take(steps + 1)
    it = iter(ds)
    m.train_on_batch(*next(it))                      # warm-up / graph build
    t0 = time.perf_counter()
    for xb, yb in it:
        m.train_on_batch(xb, yb)
    return (time.perf_counter() - t0) / steps

steps_per_epoch = int(np.ceil(len(splits.load_split("train")) / config.BATCH_SIZE))
s1 = time_steps(model)
probe, probe_backbone = resnet50.build_model(num_classes)
resnet50.unfreeze_upper_layers(probe_backbone)
resnet50.compile_model(probe, resnet50.TRAIN["stage2_lr"])
s2 = time_steps(probe)
del probe, probe_backbone
print(f"{steps_per_epoch} steps/epoch at batch {config.BATCH_SIZE}")
print(f"stage 1: {s1:.2f} s/step  -> ~{s1 * steps_per_epoch / 60:.0f} min/epoch (+ validation)")
print(f"stage 2: {s2:.2f} s/step  -> ~{s2 * steps_per_epoch / 60:.0f} min/epoch (+ validation)")

1317 steps/epoch at batch 32
stage 1: 2.09 s/step  -> ~46 min/epoch (+ validation)
stage 2: 2.26 s/step  -> ~50 min/epoch (+ validation)


## 6 · Smoke test of the full two-stage loop

Runs **2 epochs × 3 steps per stage** at batch 8 and writes to
`models/resnet50/smoke_test/`. This checks that training, callbacks, checkpointing
and reloading work end to end. **Its metrics say nothing about model quality.**

In [7]:
smoke_dir = resnet50.MODEL_DIR / "smoke_test"
smoke_model, smoke_hist = resnet50.train(
    stage1_epochs=2, stage2_epochs=2, batch_size=8, steps_per_epoch=3, validation_steps=2,
    model_path=smoke_dir / "best_resnet50.keras", reports_dir=smoke_dir)

ckpt = smoke_dir / "best_resnet50.keras"
assert ckpt.exists()
restored = keras.models.load_model(ckpt)
assert restored.output_shape == (None, num_classes)
assert list(np.load(smoke_dir / "class_names.npy")) == class_names
for f in sorted(smoke_dir.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.2f} MB")
print("Smoke test passed: both stages ran, checkpoint saved and reloaded.")

Stage 1 params: {'trainable': 534310, 'non_trainable': 23587712}
Epoch 1/2


C:\Users\H P\Desktop\Study\7 - Sem\Project\new smart crop care\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(



Epoch 1: val_accuracy improved from None to 0.00000, saving model to C:\Users\H P\Desktop\Study\7 - Sem\Project\new smart crop care\models\resnet50\smoke_test\best_resnet50.keras



Epoch 1: finished saving model to C:\Users\H P\Desktop\Study\7 - Sem\Project\new smart crop care\models\resnet50\smoke_test\best_resnet50.keras


3/3 - 11s - 4s/step - accuracy: 0.0417 - loss: 4.7545 - top3: 0.0833 - val_accuracy: 0.0000e+00 - val_loss: 4.1052 - val_top3: 0.0000e+00 - learning_rate: 0.0010


...

Epoch 2/2



Epoch 2: val_accuracy did not improve from 0.00000


3/3 - 4s - 1s/step - accuracy: 0.0833 - loss: 3.6126 - top3: 0.2917 - val_accuracy: 0.0000e+00 - val_loss: 3.2736 - val_top3: 0.0000e+00 - learning_rate: 1.0000e-05


Restoring model weights from the end of the best epoch: 1.


best_resnet50.keras            101.43 MB
class_names.npy                  0.01 MB
training_history.json            0.00 MB
training_stage1.csv              0.00 MB
training_stage2.csv              0.00 MB
Smoke test passed: both stages ran, checkpoint saved and reloaded.


## 7 · Full training

Off by default. On CPU, expect the per-epoch times estimated in section 5. When
enabled, this cell writes `models/resnet50/best_resnet50.keras`,
`models/resnet50/class_names.npy` and the per-stage logs in `artifacts/reports/`.

In [8]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    tf.keras.utils.set_random_seed(config.SEED)
    model, history = resnet50.train()
else:
    print("Skipped. Set RUN_FULL_TRAINING = True or run: python scripts/train_resnet50.py")

Skipped. Set RUN_FULL_TRAINING = True or run: python scripts/train_resnet50.py


In [9]:
hist_path = resnet50.REPORTS_DIR / "training_history.json"
if hist_path.exists() and resnet50.BEST_MODEL_PATH.exists():
    h = json.loads(hist_path.read_text())
    n1 = len(h["stage1"]["loss"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        for key, color, name in [(metric, SERIES[0], "train"), (f"val_{metric}", SERIES[1], "val")]:
            values = h["stage1"][key] + h["stage2"][key]
            ax.plot(range(1, len(values) + 1), values, color=color, linewidth=2, label=name)
        ax.axvline(n1 + 0.5, color="#b5b4ad", linestyle="--", linewidth=1)
        ax.set_title(metric); ax.set_xlabel("epoch")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / "03_training_curves.png", dpi=120)
    plt.show()
else:
    print("No full training run yet, so there are no curves or accuracy figures to report.")

No full training run yet, so there are no curves or accuracy figures to report.
